In [ ]:
import ee
import geemap
import xee
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
ee.Authenticate()
ee.Initialize(
    project = 'infinite-cache-458105-a3',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [ ]:
map = geemap.Map(basemap='TERRAIN')
map

In [ ]:
loc = map.draw_last_feature.geometry()

In [ ]:
pak = (
    ee.FeatureCollection("USDOS/LSIB/2017") 
    .filterBounds(loc)
    .geometry()
)
pak

In [ ]:
map.addLayer(pak, {}, 'PAKISTAN')

In [ ]:
pr = (
     ee.ImageCollection("NASA/GPM_L3/IMERG_MONTHLY_V07")
    .filterDate('2025', '2026')
    .select('precipitation')
    .map(lambda x: x.clip(pak).copyProperties(x, x.propertyNames()))
)
pr

In [ ]:
pr = xr.open_dataset(
    pr,
    engine='ee',
    scale=0.27,
    crs='EPSG:4326',
    geometry=pak,
)




In [ ]:
ds = pr.sortby('time')*1000
ds

In [ ]:
ds.precipitation.plot.contourf(
    x='lon',
    y='lat',
    col='time',
    col_wrap=6,
    robust=True
)
ds